In [0]:
-- Gold occupation-to-skill bridge
-- Grain: one row per O*NET occupation, skill family, skill, and rating scale.

CREATE OR REPLACE VIEW workforce_analytics.gold.bridge_occupation_skill
COMMENT 'Combined O*NET essential and transferable skill ratings with detailed occupation titles.'
AS

WITH combined_skills AS (

    SELECT
        onet_soc_code,
        soc_code,
        'essential' AS skill_family,
        element_id,
        element_name,
        scale_id,
        data_value,
        source_file_path,
        run_id,
        processed_at
    FROM workforce_analytics.silver.onet_essential_skills
    WHERE data_value IS NOT NULL

    UNION ALL

    SELECT
        onet_soc_code,
        soc_code,
        'transferable' AS skill_family,
        element_id,
        element_name,
        scale_id,
        data_value,
        source_file_path,
        run_id,
        processed_at
    FROM workforce_analytics.silver.onet_transferable_skills
    WHERE data_value IS NOT NULL
),

ranked_skills AS (
    SELECT
        skill.onet_soc_code,
        skill.soc_code,
        occupation.occupation_title,

        skill.skill_family,
        skill.element_id AS skill_id,
        skill.element_name AS skill_name,
        skill.scale_id,

        CASE UPPER(skill.scale_id)
            WHEN 'IM' THEN 'importance'
            WHEN 'LV' THEN 'level'
            ELSE LOWER(skill.scale_id)
        END AS scale_name,

        skill.data_value AS skill_score,

        DENSE_RANK() OVER (
            PARTITION BY
                skill.onet_soc_code,
                skill.skill_family,
                skill.scale_id
            ORDER BY
                skill.data_value DESC,
                skill.element_name ASC
        ) AS skill_rank,

        skill.source_file_path,
        skill.run_id,
        skill.processed_at

    FROM combined_skills AS skill

    LEFT JOIN workforce_analytics.silver.onet_occupations AS occupation
        ON skill.onet_soc_code = occupation.onet_soc_code
)

SELECT *
FROM ranked_skills;
